In [20]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
from Train_fun import train_fun
from scipy.interpolate import Rbf
from RK import solve_rk4_adaptive_fixed_dt
from RK import build_f_sparse_mixed
from Bulid_Library import build_polynomial_library
from SSSR import SSSR
from SSSR import linear_reg

In [3]:
def rrmse(x,y):
    return (np.mean((x-y)**2))**0.5/(np.mean(y**2))**0.5

# Data generation

In [36]:
def Burgers_data(alpha, a, L, nx, T, nt):
    dx = L / (nx - 1)
    x = np.linspace(0, L, nx)
    dt = T / (nt - 1)                  
    
    # 初始条件 
    u = a*np.sin(x)       # 例：u(x,0)=sin(x)

    U = np.zeros((nt, nx))
    U[0, :] = u.copy()
    
    # ——— 周期性边界 + 中心差分更新 ———
    for k in range(nt-1):
        u_x  = (np.roll(u, -1) - np.roll(u, 1))   / (2*dx)
        u_xx = (np.roll(u, -1) - 2*u + np.roll(u, 1)) / dx**2
        u = u + dt * (-u*u_x + alpha*u_xx)
        U[k+1, :] = u
    
    return U

In [50]:
m = 401
mu1_number = 6
mu2_number = 6
Para_number = mu1_number * mu2_number
Para = np.array(np.meshgrid(np.linspace(0.2, 0.7, mu1_number), np.linspace(0.5, 0.9, mu2_number))).T.reshape(-1,2)
U_all = []

for point in Para:
    mu1,mu2 = point[0],point[1]
    U_alpha = Burgers_data(mu1, mu2, 2*np.pi, 201, 4, 10001)
    U_alpha = U_alpha[::25,:]
    U_all.append(U_alpha)

U = np.vstack(U_all)  

# POD

In [51]:
U_mean = np.mean(U, axis=0)
U_centered = U - U_mean
U_snapshots = U_centered.T  
Phi, Sigma, Vt = np.linalg.svd(U_snapshots, full_matrices=False)

latent_dim = 3
Phi_r = Phi[:,:latent_dim]
Z = np.transpose(np.dot(Phi_r.T, U_snapshots))  # shape = (m, r)
U_reconstructed = np.dot(Z, Phi_r.T) + U_mean
print('reconstruct error:', rrmse(U_reconstructed,U))

reconstruct error: 0.001273185299241168


# SSSR 

In [52]:
dt = 4/(m-1)
dZ = np.zeros_like(Z)

for i in range(Para_number):
    start = i * m
    end = (i + 1) * m
    a_seg = Z[start:end, :]  # shape: (m, r)
    
    da_seg = np.zeros_like(a_seg)
    da_seg[1:-1] = (a_seg[2:] - a_seg[:-2]) / (2 * dt)
    da_seg[0] = (a_seg[1] - a_seg[0]) / dt
    da_seg[-1] = (a_seg[-1] - a_seg[-2]) / dt
    
    dZ[start:end, :] = da_seg

In [53]:
include_functions = False
include_bias=False
degree = 2
Theta, feature_names = build_polynomial_library(Z, degree=degree, include_bias=include_bias, include_functions=include_functions)
sparsity_level = 4 
Z_pred = Z.copy()
coef_matrix = np.zeros((Para_number,(sparsity_level+1)*latent_dim))
Supports = np.zeros((latent_dim,sparsity_level))

for k in range(latent_dim):
    loss = 0
    support = SSSR(Theta, dZ[:,k].reshape(-1,1), sparsity_level=sparsity_level, Para_number=Para_number)
    print('Support set for latent variable {}:'.format(k), support)
    Supports[k,:] = support
    Library = []
    for j in range(Para_number):
        Target = dZ[m*j:m*(j+1),k].reshape(-1,1)
        state = Z[m*j:m*(j+1),k].reshape(-1,1)
        Feature = Theta[m*j:m*(j+1),support]
        reg, pred = linear_reg(Feature,Target)
        coef_matrix[j,(sparsity_level+1)*k] = reg.intercept_
        coef_matrix[j,(sparsity_level+1)*k+1:(sparsity_level+1)*(k+1)] = reg.coef_
        loss = loss + rrmse(pred,Target)
        state_next_pred = state + dt * pred
        state_pred = state.copy()
        state_pred[1:] = state_next_pred[:-1]
        Library.append(state_pred)
    print('Regression error of latent variable {}:'.format(k), loss/Para_number)
    state_pred = np.vstack(Library)
    Z_pred[:,k] = np.squeeze(state_pred)
    Supports = Supports.astype('int')

Support set for latent variable 0: [0, 1, 4, 2]
Regression error of latent variable 0: 0.00018877185485728618
Support set for latent variable 1: [1, 3, 0, 6]
Regression error of latent variable 1: 0.0023780060775241366
Support set for latent variable 2: [4, 2, 0, 1]
Regression error of latent variable 2: 0.01322444562886864


In [54]:
Library = []
for j in range(Para_number):
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=coef_matrix[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.00001)
    Library.append(Y)
Z_pred_multi_step = np.vstack(Library)

In [55]:
print('The prediction error of Z by multiple step:', rrmse(Z_pred_multi_step,Z))
U_pred_multi_step = np.dot(Z_pred_multi_step, Phi_r.T) + U_mean  # shape = (m, n)
print('The prediction error of X by multiple step:', rrmse(U_pred_multi_step,U)) 

The prediction error of Z by multiple step: 0.0003384319913779275
The prediction error of X by multiple step: 0.0012847958312106951


# Parameter-to-coefficient mapping

In [56]:
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.interpolate import Rbf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [57]:
Para_data = np.hstack((Para,coef_matrix))

## RBF

In [58]:
X = Para_data[:, 0:2]        
Y = Para_data[:, 2:] 
N, r = Y.shape

# -----------------------------
# 2. 为每个低维系数单独构建 RBF 插值器
# -----------------------------
rbf_models = []
for j in range(r):
    rbf = Rbf(X[:,0], X[:,1], Y[:, j], function='multiquadric')
    rbf_models.append(rbf)

## NN

In [60]:
X = Para_data[:, 0:2]        
Y = Para_data[:, 2:]        

X_train_t = torch.from_numpy(X).float().to(device)
Y_train_t = torch.from_numpy(Y).float().to(device)
output_dim = Y.shape[1]

In [61]:
class FourNet_sin(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, 128)
        self.linear2 = nn.Linear(128, 64)
        self.linear3 = nn.Linear(64,64)
        self.linear4 = nn.Linear(64,output_dim)

    def forward(self, x):
        x = torch.sin(self.linear1(x))
        x = torch.sin(self.linear2(x))
        x = torch.sin(self.linear3(x))
        x = self.linear4(x)
        return x

# 实例化模型
model = FourNet_sin(input_dim=2, output_dim=output_dim).to(device)

In [62]:
loss,time,loss_his = train_fun(model,X_train_t,Y_train_t,N_red_lr=4,epochs=15000,lr=0.001,threshold=0.0001,printfun=False)

运行时间： 56.539067029953
loss: 0.000608590489719063


# Online predict 

## Set test parameter point

In [74]:
Para_test = np.array([[0.34,0.71]])
Para_test_number = len(Para_test)

U_test_all = []

for point in Para_test:
    mu1,mu2 = point[0],point[1]
    U_alpha = Burgers_data(mu1, mu2, 2*np.pi, 201, 4, 10001)
    U_alpha = U_alpha[::25,:]
    U_test_all.append(U_alpha)

U_test = np.vstack(U_test_all)  

In [86]:
# get latent initial condiction
U_centered_test = U_test - U_mean
U_snapshots_test = U_centered_test.T 
Z_test = np.dot(Phi_r.T, U_snapshots_test).T

## RBF

In [84]:
# predcit the coefficients
Y_test_pred_RBF = []
for j, rbf in enumerate(rbf_models):
    Y_test_pred_RBF.append(rbf(Para_test[:,0],Para_test[:,1]).reshape(-1,1))
Y_test_pred_RBF = np.hstack(Y_test_pred_RBF)

# sloving the latent dynamical system 
Library = []
for j in range(Para_test_number):   
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=Y_test_pred_RBF[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z_test[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.001)
    Library.append(Y)
Z_pred_multi_step_RBF = np.vstack(Library)

# reconstruction 
U_pred_multi_step_RBF = np.dot(Z_pred_multi_step_RBF, Phi_r.T) + U_mean  # shape = (m, n)

In [85]:
rrmse(U_pred_multi_step_RBF,U_test)

0.0018587939493673422

## NN

In [89]:
# predcit the coefficients
model.eval()
with torch.no_grad():
    Y_test_pred_NN = model(torch.from_numpy(Para_test).float().to(device)).cpu().numpy()

# sloving the latent dynamical system 
Library = []
for j in range(Para_test_number):   
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=Y_test_pred_NN[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z_test[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.001)
    Library.append(Y)
Z_pred_multi_step_NN = np.vstack(Library)

# reconstruction 
U_pred_multi_step_NN= np.dot(Z_pred_multi_step_NN, Phi_r.T) + U_mean  # shape = (m, n)

In [90]:
rrmse(U_pred_multi_step_NN,U_test)

0.0011472505717936082